# Semana 11: Determinantes, Inversas y Rango
## Notebook de Ejercicios (NB2) - Análisis de Rango en Imágenes

### Propósito de la sesión
Aplicar los conceptos de **rango de una matriz** a un problema real: imágenes digitales. Trabajaremos con imágenes en escala de grises, que son matrices de píxeles, y exploraremos cómo el rango se relaciona con la estructura de la imagen y cómo cambia al añadir ruido.

### Objetivos de aprendizaje
*   Cargar una imagen real (desde librerías o URL) y convertirla a escala de grises.
*   Interpretar una imagen como una matriz de píxeles.
*   Calcular el **rango** de la matriz de la imagen y relacionarlo con la complejidad de la imagen.
*   Añadir **ruido** a la imagen y observar cómo cambia el rango.
*   Reflexionar sobre el significado del rango en el contexto de procesamiento de imágenes y machine learning.

## Configuración Inicial

Importamos las librerías necesarias: numpy, matplotlib, skimage para imágenes, y scipy para procesamiento.

In [ ]:
# Importamos librerías
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, color, util
from scipy import ndimage

# Configuración de matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 8)

# Fijamos semilla para reproducibilidad
np.random.seed(42)

print("🎯 Librerías importadas correctamente.")

---
## 1. Carga de una imagen desde librerías

Utilizaremos imágenes incluidas en `skimage.data` (cámara, astronauta, etc.) y también mostraremos cómo cargar una imagen desde una URL.

In [ ]:
# Cargamos una imagen de ejemplo (cámara)
image = data.camera()
print(f"Imagen original - shape: {image.shape}, dtype: {image.dtype}")

# Mostramos la imagen
plt.figure(figsize=(8, 8))
plt.imshow(image, cmap='gray')
plt.title('Imagen original (cámara)')
plt.axis('off')
plt.show()

# Opcional: cargar otra imagen de skimage
image2 = data.astronaut()
image2_gray = color.rgb2gray(image2)
print(f"\nImagen astronauta (escala de grises) - shape: {image2_gray.shape}")

plt.figure(figsize=(8, 8))
plt.imshow(image2_gray, cmap='gray')
plt.title('Imagen astronauta (escala de grises)')
plt.axis('off')
plt.show()

---
## Actividad 1: Calcular el rango de la matriz de la imagen

Una imagen en escala de grises es una matriz $\mathbf{I} \in \mathbb{R}^{h \times w}$. Vamos a calcular su **rango** usando `np.linalg.matrix_rank`.

In [ ]:
def analyze_image_rank(image, name="Imagen"):
    """
    Calcula el rango de una imagen y muestra estadísticas.
    """
    # Aseguramos que la imagen sea 2D (escala de grises)
    if image.ndim == 3:
        image = color.rgb2gray(image)
    
    h, w = image.shape
    
    # Calculamos el rango
    rank = np.linalg.matrix_rank(image)
    
    # Rango máximo posible es min(h, w)
    max_rank = min(h, w)
    
    print(f"🔷 {name}:")
    print(f"  Dimensiones: {h} x {w}")
    print(f"  Rango de la matriz: {rank}")
    print(f"  Rango máximo posible: {max_rank}")
    print(f"  Proporción rank/max: {rank / max_rank:.4f}")
    
    return rank, max_rank

# Analizamos la imagen 'camera'
rank_cam, max_cam = analyze_image_rank(image, "Imagen camera")

print("\n" + "-"*50)
# Analizamos la imagen 'astronauta' (ya en grises)
rank_ast, max_ast = analyze_image_rank(image2_gray, "Imagen astronauta")

### 1.1. Interpretación

Observamos que el rango es mucho menor que el rango máximo posible ($h$ o $w$). Esto se debe a que las imágenes naturales tienen mucha **estructura** y **redundancia**: las filas y columnas no son linealmente independientes; hay correlaciones espaciales.

Por ejemplo, una imagen completamente uniforme tendría rango 1 (todas las filas iguales, todas las columnas iguales). Cuanto más compleja es la imagen, mayor es el rango, pero aún así muy por debajo del máximo.

In [ ]:
# Demostración: imagen de ruido aleatorio (blanco) - tendría rango alto
random_image = np.random.rand(100, 100)
rank_random = np.linalg.matrix_rank(random_image)
print(f"Imagen de ruido aleatorio 100x100 - Rango: {rank_random} (máximo 100)")

# Imagen constante
constant_image = np.ones((100, 100)) * 128
rank_const = np.linalg.matrix_rank(constant_image)
print(f"Imagen constante 100x100 - Rango: {rank_const}")

# Visualizamos
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.imshow(random_image, cmap='gray')
plt.title(f'Ruido aleatorio (rango {rank_random})')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(constant_image, cmap='gray')
plt.title(f'Imagen constante (rango {rank_const})')
plt.axis('off')
plt.show()

---
## Actividad 2: Añadir ruido y observar cómo cambia el rango

El ruido añade información aleatoria que puede **aumentar** el rango de la imagen, haciendo que las filas/columnas sean más independientes. Veremos este efecto.

In [ ]:
def add_noise_and_analyze(image, noise_levels=[0.01, 0.05, 0.1, 0.2], name="Imagen"):
    """
    Añade ruido gaussiano con diferentes niveles y observa el rango.
    """
    if image.ndim == 3:
        image = color.rgb2gray(image)
    
    # Normalizamos la imagen a [0,1] para el ruido
    if image.max() > 1.0:
        image = image / 255.0
    
    original_rank = np.linalg.matrix_rank(image)
    max_rank = min(image.shape)
    
    print(f"\n🔶 {name} - Rango original: {original_rank} / {max_rank}")
    
    ranks = [original_rank]
    levels = [0] + noise_levels
    
    plt.figure(figsize=(14, 8))
    
    for i, sigma in enumerate(levels[1:], 1):
        # Añadir ruido gaussiano
        noisy = util.random_noise(image, mode='gaussian', var=sigma**2, seed=42)
        rank_noisy = np.linalg.matrix_rank(noisy)
        ranks.append(rank_noisy)
        
        plt.subplot(2, len(levels)-1, i)
        plt.imshow(noisy, cmap='gray')
        plt.title(f'σ={sigma}, rango={rank_noisy}')
        plt.axis('off')
    
    plt.suptitle(f'Efecto del ruido en el rango - {name}', fontsize=14)
    plt.tight_layout()
    plt.show()
    
    # Gráfico de evolución del rango
    plt.figure(figsize=(10, 6))
    plt.plot(levels, ranks, 'bo-', linewidth=2, markersize=8)
    plt.xlabel('Nivel de ruido σ')
    plt.ylabel('Rango')
    plt.title(f'Evolución del rango con el ruido - {name}')
    plt.grid(True)
    plt.axhline(y=max_rank, color='r', linestyle='--', label=f'Rango máximo ({max_rank})')
    plt.legend()
    plt.show()
    
    return ranks

# Aplicamos a la imagen 'camera'
ranks_cam = add_noise_and_analyze(image, noise_levels=[0.01, 0.05, 0.1, 0.2], name="Imagen camera")

# Aplicamos a la imagen 'astronauta'
ranks_ast = add_noise_and_analyze(image2_gray, noise_levels=[0.01, 0.05, 0.1, 0.2], name="Imagen astronauta")

### 2.1. Análisis de resultados

Observamos que:
*   La imagen original tiene un rango relativamente bajo debido a su estructura.
*   Al añadir ruido, el rango **aumenta**.
*   Con suficiente ruido, el rango puede acercarse al máximo posible (min(h, w)).
*   Esto ocurre porque el ruido rompe las dependencias lineales entre filas y columnas.

**Interpretación en machine learning:**
En datasets reales, las características suelen estar correlacionadas, lo que da lugar a matrices de datos con rango bajo (o deficiente). Esto puede causar problemas de multicolinealidad en regresión. El ruido (o la regularización) puede aumentar el rango efectivo, pero también introduce perturbaciones.

---
## 3. Experimentos adicionales

### 3.1. Efecto del tamaño de la imagen en el rango

El rango máximo posible es el mínimo entre alto y ancho. Veamos cómo cambia el rango al redimensionar la imagen.

In [ ]:
from skimage.transform import resize

def analyze_scaled_ranks(image, scales=[0.25, 0.5, 1.0, 2.0], name="Imagen"):
    """
    Redimensiona la imagen y calcula el rango.
    """
    if image.ndim == 3:
        image = color.rgb2gray(image)
    
    h, w = image.shape
    
    print(f"\n🔷 {name} - Análisis de escalado:")
    print(f"{'Escala':<10} {'Dimensiones':<20} {'Rango':<10} {'Rango máx':<10} {'Proporción':<10}")
    print("-" * 60)
    
    for scale in scales:
        new_h, new_w = int(h * scale), int(w * scale)
        img_resized = resize(image, (new_h, new_w), preserve_range=True, anti_aliasing=True)
        rank = np.linalg.matrix_rank(img_resized)
        max_rank = min(new_h, new_w)
        proportion = rank / max_rank
        print(f"{scale:<10} {new_h}x{new_w:<16} {rank:<10} {max_rank:<10} {proportion:.4f}")

analyze_scaled_ranks(image, scales=[0.25, 0.5, 1.0], name="Imagen camera")
analyze_scaled_ranks(image2_gray, scales=[0.25, 0.5, 1.0], name="Imagen astronauta")

### 3.2. Rango de la matriz después de aplicar un filtro (suavizado)

El suavizado (filtro Gaussiano) reduce las altas frecuencias y puede **disminuir** el rango, ya que hace que filas y columnas sean más similares.

In [ ]:
def apply_filter_and_analyze(image, sigma=2.0):
    """Aplica un filtro Gaussiano y analiza el rango."""
    if image.ndim == 3:
        image = color.rgb2gray(image)
    
    original_rank = np.linalg.matrix_rank(image)
    
    # Aplicar filtro Gaussiano
    smoothed = ndimage.gaussian_filter(image, sigma=sigma)
    smoothed_rank = np.linalg.matrix_rank(smoothed)
    
    print(f"Rango original: {original_rank}")
    print(f"Rango después de filtro Gaussiano (σ={sigma}): {smoothed_rank}")
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    ax1.imshow(image, cmap='gray')
    ax1.set_title(f'Original (rango {original_rank})')
    ax1.axis('off')
    
    ax2.imshow(smoothed, cmap='gray')
    ax2.set_title(f'Suavizado σ={sigma} (rango {smoothed_rank})')
    ax2.axis('off')
    plt.show()
    
    return smoothed_rank

print("🔷 Imagen camera:")
apply_filter_and_analyze(image, sigma=2.0)

print("\n🔶 Imagen astronauta:")
apply_filter_and_analyze(image2_gray, sigma=2.0)

---
## Ejercicios para el Estudiante

1.  **Imagen propia:** Sube una imagen propia (puedes usar `from google.colab import files` para subir un archivo) y repite el análisis de rango y ruido.

2.  **Comparación de imágenes:** Elige tres imágenes de diferentes tipos (paisaje, retrato, imagen de texto, etc.) y compara sus rangos. ¿Qué tipo de imagen tiende a tener mayor rango? ¿Por qué?

3.  **Ruido sal y pimienta:** En lugar de ruido gaussiano, prueba con ruido **sal y pimienta** (`mode='s&p'` en `util.random_noise`). ¿Cómo afecta al rango?

4.  **Compresión y rango:** Toma una imagen y aplícale compresión JPEG (puedes guardar con baja calidad y volver a cargar). ¿Cómo cambia el rango? Relaciona con pérdida de información.

5.  **Reflexión:** En una red neuronal, si la matriz de pesos de una capa tiene rango bajo, significa que la capa está proyectando las entradas en un espacio de menor dimensión. ¿Cómo podría esto relacionarse con la **regularización** o con la **extracción de características**?

---
## Conclusiones de la Sesión

*   Una **imagen digital en escala de grises** es una matriz de píxeles. Su **rango** mide la complejidad lineal de sus filas/columnas.
*   Las imágenes naturales tienen rango **mucho menor** que el máximo posible debido a correlaciones espaciales y redundancia.
*   Al añadir **ruido**, el rango **aumenta** porque se rompen las dependencias lineales. Con suficiente ruido, el rango puede acercarse al máximo.
*   El **suavizado** (filtro Gaussiano) reduce el rango, haciendo la imagen más uniforme.
*   Estos conceptos son relevantes en machine learning: matrices de datos con rango bajo pueden indicar multicolinealidad, mientras que añadir ruido (o regularización) puede aumentar el rango efectivo.
*   El análisis de rango es útil para entender la **capacidad representacional** de capas en redes neuronales y para detectar redundancias.

¡Ahora sabes cómo el rango de una matriz revela información sobre la estructura de una imagen y cómo el ruido la modifica!